# Primera practica de ARTEFACTOS 

* Realizado por : AYA DAHMANI .
* Asignatura : systemas intelegentes.


Este notebook contiene **todo el proyecto completo y autocontenido** en un solo archivo `.ipynb`. Al ejecutar la celda principal, se abrirá la ventana gráfica con la estética de exploradores y reliquias efímeras.

In [ ]:
import pygame
import random
import sys

# 1. CONFIGURACIÓN Y MODELO DEL JUEGO 
RELICT_TYPES = {
    1: {"nombre": "Amuleto de Jade", "valor": 10, "vida_min": 12, "vida_max": 15},
    2: {"nombre": "Pergamino Ancestral", "valor": 20, "vida_min": 10, "vida_max": 13},
    3: {"nombre": "Máscara Solar", "valor": 30, "vida_min": 8, "vida_max": 11},
    4: {"nombre": "Cetro Perdido", "valor": 40, "vida_min": 6, "vida_max": 9},
    5: {"nombre": "Corazón de Atlantis", "valor": 50, "vida_min": 5, "vida_max": 7}
}

def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def get_valid_moves(pos, map_size=10):
    moves = {
        'ARRIBA': (-1, 0),
        'ABAJO': (1, 0),
        'IZQUIERDA': (0, -1),
        'DERECHA': (0, 1),
        'PERMANECER': (0, 0)
    }
    valid = {}
    for name, (dr, dc) in moves.items():
        nr, nc = pos[0] + dr, pos[1] + dc
        if 0 <= nr < map_size and 0 <= nc < map_size:
            valid[name] = (nr, nc)
    return valid

class Tablerojuego:
    def __init__(self, map_size=10):
        self.map_size = map_size
        self.turn = 0
        self.max_turns = 100
        self.pos_a = (0, 0)                  # Jugador A (Humano)
        self.pos_b = (map_size - 1, map_size - 1)  # Jugador B (IA)
        self.score_a = 0
        self.score_b = 0
        self.relics = {}
        self._init_relics()

    def _init_relics(self):
        occupied = {self.pos_a, self.pos_b}
        for r_id in range(1, 11):
            while True:
                pos = (random.randint(0, self.map_size - 1), random.randint(0, self.map_size - 1))
                if pos not in occupied:
                    occupied.add(pos)
                    break
            r_info = RELICT_TYPES[random.choice(list(RELICT_TYPES.keys()))]
            self.relics[r_id] = {
                "pos": pos,
                "tipo": r_info["nombre"],
                "valor": r_info["valor"],
                "lifespan": random.randint(r_info["vida_min"], r_info["vida_max"])
            }

    def is_terminal(self):
        return len(self.relics) == 0 or self.turn >= self.max_turns

def AIAction(state):
    valid_moves = get_valid_moves(state.pos_b, state.map_size)
    if not state.relics:
        m = random.choice(list(valid_moves.keys()))
        return valid_moves[m]
    best_pos = list(valid_moves.values())[0]
    best_score = -float('inf')
    for next_pos in valid_moves.values():
        score = 0
        for relic in state.relics.values():
            dist = manhattan_distance(next_pos, relic['pos'])
            if dist == 0:
                utility = relic['valor'] * 2
            elif dist >= relic['lifespan']:
                utility = -50
            else:
                utility = relic['valor'] / (dist + 1)
            score += utility
        if score > best_score:
            best_score = score
            best_pos = next_pos
    return best_pos

print("Estructura del juego lista.")

pygame 2.5.2 (SDL 2.28.3, Python 3.10.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
Estructura del juego lista.


In [ ]:
# --- 2. LANZAR INTERFAZ VISUAL (UI y Utiles) --- 
def run_game():
    pygame.init()
    WIDTH, HEIGHT = 900, 700
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("CAPTURA DE ARTEFACTOS TEMPORALES")
    
    BG_COLOR = (11, 37, 56)
    GRID_COLOR = (30, 70, 100)
    TEXT_COLOR = (255, 255, 255)
    PANEL_COLOR = (18, 52, 77)
    
    font_title = pygame.font.SysFont("Arial", 26, bold=True)
    font_sub = pygame.font.SysFont("Arial", 14)
    font_bold = pygame.font.SysFont("Arial", 15, bold=True)
    
    clock = pygame.time.Clock()
    game = Tablerojuego(map_size=10)
    
    grid_size = 10
    cell_size = 45
    grid_origin_x = 225
    grid_origin_y = 130
    
    running = True
    while running and not game.is_terminal():
        screen.fill(BG_COLOR)
        
        title = font_title.render("CAPTURA DE ARTEFACTOS TEMPORALES", True, TEXT_COLOR)
        screen.blit(title, (WIDTH // 2 - title.get_width() // 2, 20))
        sub = font_sub.render("Una competición estratégica por reliquias que desaparecen con el tiempo", True, (180, 210, 230))
        screen.blit(sub, (WIDTH // 2 - sub.get_width() // 2, 55))
        
        info = font_bold.render(f"Turno: {game.turn + 1}/{game.max_turns}   |   Humano (A): {game.score_a} pts   |   IA (B): {game.score_b} pts", True, (255, 220, 100))
        screen.blit(info, (WIDTH // 2 - info.get_width() // 2, 88))
        
        # Personaje A (Izquierda)
        pygame.draw.rect(screen, (139, 69, 19), (60, 250, 50, 100))
        pygame.draw.circle(screen, (245, 222, 179), (85, 230), 20)
        pygame.draw.rect(screen, (100, 50, 10), (70, 205, 30, 25))
        screen.blit(font_bold.render("Humano (A)", True, (255, 100, 100)), (45, 360))
        
        # Personaje B (Derecha)
        pygame.draw.rect(screen, (90, 110, 120), (790, 250, 50, 100))
        pygame.draw.circle(screen, (245, 222, 179), (815, 230), 20)
        pygame.draw.polygon(screen, (60, 70, 80), [(795, 215), (835, 215), (815, 190)])
        screen.blit(font_bold.render("IA (B)", True, (100, 200, 255)), (800, 360))
        
        # Tablero
        board_rect = pygame.Rect(grid_origin_x, grid_origin_y, grid_size * cell_size, grid_size * cell_size)
        pygame.draw.rect(screen, PANEL_COLOR, board_rect)
        pygame.draw.rect(screen, (100, 180, 220), board_rect, 2)
        
        for r in range(grid_size + 1):
            pygame.draw.line(screen, GRID_COLOR, (grid_origin_x, grid_origin_y + r * cell_size), (grid_origin_x + grid_size * cell_size, grid_origin_y + r * cell_size), 1)
        for c in range(grid_size + 1):
            pygame.draw.line(screen, GRID_COLOR, (grid_origin_x + c * cell_size, grid_origin_y), (grid_origin_x + c * cell_size, grid_origin_y + grid_size * cell_size), 1)
            
        # Reliquias
        for r_id, relic in game.relics.items():
            r, c = relic['pos']
            cx = grid_origin_x + c * cell_size + cell_size // 2
            cy = grid_origin_y + r * cell_size + cell_size // 2
            val = relic['valor']
            color = (0, 220, 255) if val == 50 else (255, 215, 0) if val == 40 else (210, 125, 45) if val == 30 else (100, 200, 100)
            pygame.draw.circle(screen, color, (cx, cy), 15)
            screen.blit(font_bold.render(str(val), True, (255, 255, 255)), (cx - 8, cy + 18))
            
        # Jugadores en tablero
        ar, ac = game.pos_a
        pygame.draw.circle(screen, (220, 50, 50), (grid_origin_x + ac * cell_size + cell_size//2, grid_origin_y + ar * cell_size + cell_size//2), 14)
        br, bc = game.pos_b
        pygame.draw.circle(screen, (50, 150, 255), (grid_origin_x + bc * cell_size + cell_size//2, grid_origin_y + br * cell_size + cell_size//2), 14)
        
        footer = font_sub.render("Usa las Flechas o WASD para moverte. [ESPACIO] Permanecer. [ESC] Salir.", True, (200, 220, 240))
        screen.blit(footer, (WIDTH // 2 - footer.get_width() // 2, 650))
        
        pygame.display.flip()
        
        action_chosen = None
        valid_moves = get_valid_moves(game.pos_a, game.map_size)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return
                elif event.key in (pygame.K_UP, pygame.K_w): action_chosen = valid_moves.get('ARRIBA')
                elif event.key in (pygame.K_DOWN, pygame.K_s): action_chosen = valid_moves.get('ABAJO')
                elif event.key in (pygame.K_LEFT, pygame.K_a): action_chosen = valid_moves.get('IZQUIERDA')
                elif event.key in (pygame.K_RIGHT, pygame.K_d): action_chosen = valid_moves.get('DERECHA')
                elif event.key == pygame.K_SPACE: action_chosen = valid_moves.get('PERMANECER')
                
                if action_chosen:
                    game.pos_a = action_chosen
                    game.pos_b = AIAction(game)
                    
                    captured = []
                    for r_id, r in game.relics.items():
                        c = False
                        if game.pos_a == r['pos']:
                            game.score_a += r['valor']
                            c = True
                        if game.pos_b == r['pos']:
                            if not c: game.score_b += r['valor']
                            c = True
                        if c: captured.append(r_id)
                    for r_id in captured: del game.relics[r_id]
                    
                    for r_id in list(game.relics.keys()):
                        game.relics[r_id]['lifespan'] -= 1
                        if game.relics[r_id]['lifespan'] <= 0: del game.relics[r_id]
                    game.turn += 1
        clock.tick(30)
        
    screen.fill(BG_COLOR)
    end = font_title.render("¡PARTIDA FINALIZADA!", True, (255, 220, 100))
    sc = font_bold.render(f"Final -> Humano (A): {game.score_a} | IA (B): {game.score_b}", True, TEXT_COLOR)
    screen.blit(end, (WIDTH // 2 - end.get_width() // 2, 250))
    screen.blit(sc, (WIDTH // 2 - sc.get_width() // 2, 320))
    pygame.display.flip()
    pygame.time.wait(4000)
    pygame.quit()

print("Función de ejecución lista. Ready to run.")

Función de ejecución lista. Ejecuta la siguiente celda para iniciar el juego.


In [4]:
# Ejecutar la interfaz visual
run_game()